In [1]:
import pickle
import pandas
import os
from Tools.DatasetTools.DatasetOperator import Dataset, DatasetTester

In [2]:
target_case = 'EF_nmhcp'

In [3]:
ModelName = 'Random Forest'

In [4]:
import logging
Logger = logging.getLogger()
Logger.setLevel(logging.INFO)
logging.basicConfig(format='%(message)s')

In [5]:
ModelName = 'Random Forest'
namefile = ModelName.replace(' ', '')
suffix = f"no_hcp_bcc_fcc_{namefile}" #'CV_restart_folds_inloop''CV_restart_folds_inloop

In [6]:
dataset = 'Fe-Mo'

In [7]:
DS = Dataset('Fe-Mo', target_name=target_case, remove_phases_query='Phase != "bcc" and Phase != "fcc" and Phase !="hcp"')

In [8]:
feature_concat_resul_loc = os.path.join(dataset, 'results', f'concatenation_results_{target_case}_{suffix}.pkl')  
if os.path.exists(feature_concat_resul_loc):
    with open(feature_concat_resul_loc, 'rb') as pkl:
        savedFCresults = pickle.load(pkl)
else:
    savedFCresults = {}

In [9]:
cleanfcresults = {}
for combi, fcresults in savedFCresults.items():
    ncurves = len(fcresults)
    if ModelName not in combi:
        Logger.info(f'{combi} is not {ModelName}')
        continue
    if ncurves < 1:
        Logger.info(f'{combi} has no curves')
        continue
    Logger.info(f'combi: {combi}, curves: {ncurves}')
    cleanfcresults[combi] = []
    for i, curve in enumerate(fcresults):
        if 'random' in curve.index:
            random_in = curve.index.get_loc('random')
        else:
            random_in = len(curve.index)
        cleancurve = curve.iloc[:,:random_in]
        nfeats = cleancurve.shape[0]
        if nfeats > 0:
            cleanfcresults[combi].append(curve)
            Logger.info(f'curve {i}, nselected = {curve.shape}, random in: {random_in}, features clean: {nfeats}')

combi: ('Random Forest', 'atomic'), curves: 5
curve 0, nselected = (8, 5), random in: 8, features clean: 8
curve 1, nselected = (8, 5), random in: 8, features clean: 8
curve 2, nselected = (9, 5), random in: 5, features clean: 9
curve 3, nselected = (9, 5), random in: 8, features clean: 9
curve 4, nselected = (9, 5), random in: 8, features clean: 9
combi: ('Random Forest', 'dataset'), curves: 5
curve 0, nselected = (21, 5), random in: 21, features clean: 21
curve 1, nselected = (21, 5), random in: 21, features clean: 21
curve 2, nselected = (22, 5), random in: 20, features clean: 22
curve 3, nselected = (22, 5), random in: 21, features clean: 22
curve 4, nselected = (22, 5), random in: 21, features clean: 22
combi: ('Random Forest', 'Canonical ACE'), curves: 10
curve 0, nselected = (197, 5), random in: 197, features clean: 197
curve 1, nselected = (198, 5), random in: 198, features clean: 198
curve 2, nselected = (198, 5), random in: 2, features clean: 198
curve 3, nselected = (155, 5)

In [10]:
with open(feature_concat_resul_loc, 'wb') as f:
    pickle.dump(cleanfcresults, f)

test by learning

In [11]:
from Tools.DatasetTools.MLConveniences import filter_features
import joblib
dataset='Fe-Mo'

In [12]:
models = ['Random Forest']

In [13]:
modelnames=[model.replace(' ','') for model in models]

In [14]:
voting_regressor_files = {modelname: os.path.join(dataset, 'results', f'voting_regressor_{modelname}.pkl')
                         for modelname in modelnames}

In [15]:
voting_regressor_files

{'RandomForest': 'Fe-Mo/results/voting_regressor_RandomForest.pkl'}

In [16]:
optimal_regressors = {}
for model in modelnames:
    optimal_regressors.update(joblib.load(voting_regressor_files[model]))

In [35]:
from sklearn.model_selection import learning_curve


for ( model, featurename ), votingregressor in optimal_regressors.items():
    estimators = votingregressor.estimators
    Logger.info(f'model:{model}, featurename: {featurename}, nestimators :{len(estimators)}' )
    X = DS.Features[featurename]
    for iestimator, estimator in estimators:
        learning_curve = estimator.named_steps.feature_selection.kw_args['learning_curve']
        nfeatures = learning_curve.shape
        transformed_X =estimator.named_steps.feature_selection.transform(X)
        Logger.info(f'index:{iestimator}, transformer curve length: {nfeatures}, nfeatures stranformed: {transformed_X.shape}')


model:Random Forest, featurename: atomic, nestimators :5
index:0, transformer curve length: (8, 5), nfeatures stranformed: (262, 6)
index:1, transformer curve length: (8, 5), nfeatures stranformed: (262, 2)
index:2, transformer curve length: (5, 5), nfeatures stranformed: (262, 3)
index:3, transformer curve length: (8, 5), nfeatures stranformed: (262, 2)
index:4, transformer curve length: (8, 5), nfeatures stranformed: (262, 2)
model:Random Forest, featurename: dataset, nestimators :5
index:0, transformer curve length: (21, 5), nfeatures stranformed: (262, 17)
index:1, transformer curve length: (21, 5), nfeatures stranformed: (262, 10)
index:2, transformer curve length: (20, 5), nfeatures stranformed: (262, 10)
index:3, transformer curve length: (21, 5), nfeatures stranformed: (262, 8)
index:4, transformer curve length: (21, 5), nfeatures stranformed: (262, 11)
model:Random Forest, featurename: Canonical ACE, nestimators :9
index:0, transformer curve length: (197, 5), nfeatures stranfo

,MagpieData mean GSvolume_pa,MagpieData maximum NUnfilled,MagpieData minimum CovalentRadius,MagpieData mode MendeleevNumber,MagpieData mode Number,Mag
Fe_pv4Mo_sv20.C36-ABBBB.FM,14.863333,6.0,132.0,50.0,42.0,1
Fe_pv15Mo_sv38.R-AAAABBBBBBB.NM,14.286226,6.0,132.0,50.0,42.0,0
Fe_pv2Mo_sv11.mu-BBABB.FM,14.926923,6.0,132.0,50.0,42.0,1
Fe_pv8Mo_sv22.sigma-BBBAB.NM,14.367333,6.0,132.0,50.0,42.0,0
Fe_pv2Mo_sv11.mu-BBBBA.NM,14.926923,6.0,132.0,50.0,42.0,0
...,...,...,...,...,...,...
Fe_pv8Mo_sv16.C36-BAABB.NM,14.036667,6.0,132.0,50.0,42.0,0
Fe_pv30.sigma.FM,10.730000,4.0,132.0,55.0,26.0,1
Fe_pv6.C15.FM,10.730000,4.0,132.0,55.0,26.0,1
Mo_sv8.A15.NM,15.690000,6.0,154.0,50.0,42.0,0
